# Projet B2 · Un agent qui sait choisir son outil · ⭐⭐⭐

**Le problème** : un LLM calcule mal, ne connaît pas tes fichiers et ne sait pas le temps qu'il fait. Il répond quand même — et il invente.
**Ce qu'on construit** : une **boucle d'agent** écrite à la main, sans framework. Le modèle a une boîte à outils Python, il choisit lequel appeler, on l'exécute, on lui rend le résultat, il rédige la réponse.
**Livrable** : ce notebook complété avec au moins 4 outils (dont un de ton cru), le tableau des 6 questions de test, les 4 cas d'échec traités, et la fiche projet finale.

**Comment l'utiliser**
- Google Colab, rien à installer. Exécute chaque cellule avec `Maj + Entrée`.
- L'interrupteur `USE_MODEL` est en tête : `False` = mode démo, tout tourne **sans GPU et sans clé** (le faux modèle suit le protocole `OUTIL: nom(arguments)` dès que le prompt système le décrit, donc c'est bien **ta boucle** qui est testée) ; `True` = le petit modèle Qwen tourne dans Colab. Le reste du notebook ne change pas.
- Les cellules **« À toi »** sont des exercices : elles s'exécutent telles quelles, la vérification affiche ✅ ou ❌, la solution est cachée juste en dessous — essaie avant de l'ouvrir.
- Ce projet est la suite directe de la [séance 12 · Agents](../../../seances/seance-12-agents-et-projet-final/) : la cellule de préparation est la même.

## 0. Préparation

La cellule `llm(messages)` de la séance 12, à l'identique — une seule chose change, les branches de `llm_factice` : en mode démo, le faux modèle **suit le protocole d'outils** au lieu de bavarder. Lance-la une fois.

In [ ]:
USE_MODEL = True   # ← mets False pour tester le notebook sans modèle (réponses factices, sans GPU)

import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Mode démo : un faux LLM qui suit le protocole OUTIL: nom(arguments) ----------
_STOP_MAJ = {"Quelle", "Quel", "Quels", "Combien", "Qui", "Que", "Où", "Comment", "Donne", "Est", "Je", "Le", "La"}

def _nom_propre(question):
    """Le dernier mot qui commence par une majuscule : le nom de la ville, du Pokémon, du type..."""
    noms = [m for m in re.findall(r"\b[A-ZÉÈÀ][\wéèêàûô'-]+", question) if m not in _STOP_MAJ]
    return noms[-1] if noms else ""

def _appel_factice(question, systeme):
    """Choisit un outil d'après les mots de la question (mode démo : aucun raisonnement, des mots-clés)."""
    q = question.lower()
    if "combien font" in q or "calcul" in q or re.search(r"\d\s*[-+*/]", q):
        expression = re.search(r"[\d(][\d\s+\-*/().]*", question)
        return "OUTIL: calculer(%s)" % (expression.group().strip() if expression else "")
    if "date" in q or "quel jour" in q or "aujourd'hui" in q:
        return "OUTIL: aujourdhui()"
    if "temps" in q or "météo" in q or "température" in q or "pleut" in q or "parapluie" in q:
        return "OUTIL: meteo(%s)" % _nom_propre(question)
    if "pokémon" in q or "attaque" in q or "statistique" in q or "vitesse" in q or "points de vie" in q:
        return "OUTIL: chercher_ligne(%s)" % _nom_propre(question)
    if "aucun outil ne convient" in systeme:            # ← prompt système corrigé en section 4
        return "Aucun outil ne sert ici : je réponds de mémoire, sans garantie."
    return "OUTIL: wikipedia(%s)" % _nom_propre(question)   # ← un outil que le modèle imagine, et qui n'existe pas

def llm_factice(messages):
    """Mode démo : dès que le prompt système décrit le protocole OUTIL:, le faux modèle le suit."""
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    question = next((m["content"] for m in messages if m["role"] == "user"), "")
    dernier = messages[-1]["content"]
    if "OUTIL:" not in systeme:                          # pas de boîte à outils : il répond de tête, et il invente
        return "De tête, je dirais environ 500. (Je n'ai rien vérifié.)"
    if dernier.startswith("Résultat de l'outil"):        # l'outil a parlé : on rédige la réponse finale
        return "D'après l'outil : " + dernier.split(":", 1)[1].strip()
    if dernier.startswith("Erreur"):                     # ça a raté : une deuxième chance, puis on renonce
        deja = [m["content"] for m in messages if m["role"] == "assistant"]
        propose = _appel_factice(question, systeme)
        if "n'existe pas" in dernier and propose in deja:
            propose = "OUTIL: chercher_ligne(%s)" % _nom_propre(question)   # on se rabat sur un outil qui existe
        return propose if propose not in deja else "Je n'ai pas réussi à obtenir le résultat. " + dernier
    return _appel_factice(question, systeme)

# ---------- Le vrai modèle : petit modèle ouvert, gratuit, sans clé ----------

if USE_MODEL:
    %pip install -q transformers accelerate
    from transformers import pipeline
    _pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def llm(messages, max_new_tokens=150, temperature=0.7):
    """Envoie une liste de messages au modèle et renvoie sa réponse (du texte)."""
    if not USE_MODEL:
        return llm_factice(messages)
    if temperature == 0:      # température 0 = toujours la réponse la plus probable
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    else:
        sortie = _pipe(messages, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature)
    return sortie[0]["generated_text"][-1]["content"].strip()

print("Modèle prêt :", "Qwen2.5-0.5B-Instruct" if USE_MODEL else "mode démo (llm_factice)")

**En option : le même appel avec une API.** Sur Colab, la clé est fournie par le formateur et rangée dans les **Secrets** (icône 🔑 à gauche), jamais dans le code. Décommente la cellule ci-dessous pour remplacer `llm()` par un gros modèle dans le cloud. Tout le reste du notebook ne change pas : il n'appelle que `llm(messages)`.

In [ ]:
# --- Variante API (à décommenter si le formateur a donné une clé) ---
# %pip install -q anthropic
# from google.colab import userdata          # les Secrets de Colab
# import anthropic
# client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
#
# def llm(messages, max_new_tokens=150, temperature=0.7):
#     systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
#     autres = [m for m in messages if m["role"] != "system"]
#     rep = client.messages.create(model="claude-opus-5", max_tokens=max_new_tokens,
#                                  system=systeme or "Tu es un assistant sympa.", messages=autres)
#     return rep.content[0].text.strip()
#
# Même idée avec Mistral : pip install mistralai, clé dans userdata.get("MISTRAL_API_KEY"),
# client.chat.complete(model="mistral-small-latest", messages=messages).choices[0].message.content

Deux helpers pour tout le notebook : `demander()` (une question + un prompt système) et `verifier()` qui affiche ✅ / ❌ sans jamais planter.

In [ ]:
def demander(question, systeme="Tu es un assistant sympa qui répond en français, en 2 phrases maximum."):
    return llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}], temperature=0)

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais planter. `condition` = un booléen, ou une fonction sans argument."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as e:
        ok = False
        print(f"   (erreur pendant la vérification : {type(e).__name__} : {e})")
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))

print("Helpers prêts.")

## 1. Le problème, et le plan

Pose une question de calcul à un modèle de langue : il répond. Le souci, c'est qu'il ne calcule pas — il **prédit le mot suivant**. Le résultat sort avec le même aplomb qu'il soit juste ou faux.

In [ ]:
print("Question :", "Combien font 17 * 23 + 5 ?")
print("Le modèle seul :", demander("Combien font 17 * 23 + 5 ?"))
print("Python          :", 17 * 23 + 5)

La bonne réponse est à une ligne de Python. L'idée de l'**agent outillé** est donc simple : ne pas demander au modèle de calculer, mais de **dire quel outil il veut** — et c'est Python qui fait le travail.

```
             ┌───────────────────────────────────────────────┐
             │                                               ▼
 question ──► LLM ──► "OUTIL: calculer(17 * 23 + 5)" ──► Python exécute
                                                              │
              réponse finale ◄── LLM ◄── "Résultat de l'outil : 396" ◄┘
```

Trois pièces à écrire, et rien d'autre :

1. **les outils** — de simples fonctions Python, testées séparément (section 2) ;
2. **le protocole** — une convention d'écriture, `OUTIL: nom(arguments)`, que l'on repère avec une expression régulière (section 3) ;
3. **la boucle** — appeler le modèle, exécuter, lui rendre le résultat, recommencer, et **s'arrêter** au bout de `max_tours` (section 3).

C'est exactement ce que font les API de *function calling* des gros modèles, en JSON et avec un schéma. Ici on l'écrit à la main, pour voir le mécanisme.

## 2. La boîte à outils

Règle du jeu : **un outil est une fonction Python normale**, avec une docstring qui dit à quoi elle sert, qui prend **une chaîne de caractères** (ce que le modèle a écrit entre les parenthèses) et qui renvoie **du texte**. Jamais d'exception qui remonte : quand ça se passe mal, elle renvoie un message commençant par `Erreur :`, que le modèle pourra lire.

On les écrit et on les teste **un par un, sans agent**. Un outil qui n'a pas été testé seul ne sera jamais débogué une fois branché.

### Outil 1 : une calculatrice sûre

`eval()` sur le texte d'un modèle, c'est donner à un inconnu le droit d'exécuter du code chez toi : `eval("__import__('os').system('rm -rf /')")` fait exactement ce qu'il annonce. On utilise donc le module `ast` : il transforme le texte en **arbre**, et on décide nous-mêmes quels nœuds on accepte de calculer. Tout le reste est refusé.

In [ ]:
import ast, operator

# les seules opérations autorisées : quatre opérations, la puissance et le moins unaire
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

def _evaluer(noeud):
    """Parcourt l'arbre de l'expression et refuse tout ce qui n'est pas un calcul."""
    if isinstance(noeud, ast.Constant) and isinstance(noeud.value, (int, float)):
        return noeud.value
    if isinstance(noeud, ast.BinOp) and type(noeud.op) in _OPS:
        return _OPS[type(noeud.op)](_evaluer(noeud.left), _evaluer(noeud.right))
    if isinstance(noeud, ast.UnaryOp) and type(noeud.op) in _OPS:
        return _OPS[type(noeud.op)](_evaluer(noeud.operand))
    raise ValueError("opération interdite")

print(_evaluer(ast.parse("2 + 3 * 4", mode="eval").body))

### À toi · exercice 1 ⭐⭐ · La calculatrice qui ne plante jamais

`_evaluer` fait le calcul mais **lève** une exception quand quelque chose cloche : une exception ferait planter la boucle de l'agent. Écris `calculer(expression)` qui l'enveloppe et renvoie, à la place, un message que le modèle peut lire.

Trois cas à attraper : `ZeroDivisionError` (division par zéro), `SyntaxError` (parenthèse non fermée, texte vide, `import os`) et le reste (`ValueError` : opération interdite).

Résultat attendu : `calculer("17 * 23 + 5")` → `396`, et `calculer("import os")` → un texte qui commence par `Erreur :`.

<details><summary>Indice</summary>

`ast.parse(expression, mode="eval").body` donne la racine de l'arbre, à passer à `_evaluer`. Le tout dans un `try` suivi de trois `except`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def calculer(expression):
    """Calcule une expression arithmétique (+ - * / ** et parenthèses). Ex. calculer("17 * 23 + 5") → 396."""
    try:
        return _evaluer(ast.parse(expression, mode="eval").body)
    except ZeroDivisionError:
        return "Erreur : division par zéro, je ne peux pas."
    except SyntaxError:
        return "Erreur : ce n'est pas une expression arithmétique (parenthèse oubliée ? mot interdit ?)."
    except Exception as e:
        return f"Erreur : {e} — je ne sais faire que + - * / ** et des parenthèses."

for essai in ["17 * 23 + 5", "(2 + 3) * 4", "10 / 0", "import os", "(2 + 3", ""]:
    print(f"{essai!r:16} → {calculer(essai)}")
```

</details>

In [ ]:
# À toi
def calculer(expression):
    """Calcule une expression arithmétique (+ - * / ** et parenthèses). Ex. calculer("17 * 23 + 5") → 396."""
    return None

for essai in ["17 * 23 + 5", "(2 + 3) * 4", "10 / 0", "import os", "(2 + 3", ""]:
    print(f"{essai!r:16} → {calculer(essai)}")

In [ ]:
verifier("Exercice 1 · le calcul est juste", lambda: calculer("17 * 23 + 5") == 396)
verifier("Exercice 1 · les parenthèses sont respectées", lambda: calculer("(2 + 3) * 4") == 20)
verifier("Exercice 1 · division par zéro : un message, pas un plantage", lambda: str(calculer("10 / 0")).startswith("Erreur"))
verifier("Exercice 1 · « import os » est refusé proprement", lambda: str(calculer("import os")).startswith("Erreur"))
verifier("Exercice 1 · parenthèse non fermée", lambda: str(calculer("(2 + 3")).startswith("Erreur"))
verifier("Exercice 1 · texte vide", lambda: str(calculer("")).startswith("Erreur"))
verifier("Exercice 1 · appel de fonction refusé", lambda: str(calculer("__import__('os').getcwd()")).startswith("Erreur"))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if calculer("17 * 23 + 5") != 396:
    def calculer(expression):
        try:
            return _evaluer(ast.parse(expression, mode="eval").body)
        except ZeroDivisionError:
            return "Erreur : division par zéro, je ne peux pas."
        except SyntaxError:
            return "Erreur : ce n'est pas une expression arithmétique (parenthèse oubliée ? mot interdit ?)."
        except Exception as e:
            return f"Erreur : {e} — je ne sais faire que + - * / ** et des parenthèses."

### Outil 2 : chercher dans un fichier de données

Le deuxième outil donne au modèle accès à des données qu'il n'a jamais vues. On prend le tableau des Pokémon (800 lignes, 13 colonnes), chargé **par URL**. Si le réseau manque — Colab hors ligne, salle sans wifi — un extrait écrit en dur dans le notebook prend le relais : le reste du notebook ne s'en aperçoit pas.

In [ ]:
REPLI = """#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
4,Charmander,Fire,,309,39,52,43,60,50,65,1,False
6,Charizard,Fire,Flying,534,78,84,78,109,85,100,1,False
25,Pikachu,Electric,,320,35,55,40,50,50,90,1,False
94,Gengar,Ghost,Poison,500,60,65,60,130,75,110,1,False
130,Gyarados,Water,Flying,540,95,125,79,60,100,81,1,False
143,Snorlax,Normal,,540,160,110,65,65,110,30,1,False
150,Mewtwo,Psychic,,680,106,110,90,154,90,130,1,True
448,Lucario,Fighting,Steel,525,70,110,70,115,70,90,4,False
"""
import io

URL_POKEMON = ("https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/"
               "raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv")
try:
    POKEMON = pd.read_csv(URL_POKEMON)
    SOURCE = "URL"
except Exception as e:
    POKEMON = pd.read_csv(io.StringIO(REPLI))
    SOURCE = f"repli écrit dans le notebook ({type(e).__name__})"

print(f"{len(POKEMON)} lignes chargées depuis : {SOURCE}")
POKEMON.head(3)

### À toi · exercice 2 ⭐⭐ · Chercher une ligne dans le tableau

Écris `chercher_ligne(nom)` : elle cherche un Pokémon par son nom et renvoie **une phrase** avec ses statistiques (le modèle ne sait pas lire un `DataFrame`, il lit du texte).

Règles : la casse n'a pas d'importance, un nom vide et un nom introuvable renvoient un message `Erreur : ...`. Le fichier étant en anglais, les noms le sont aussi (`Pikachu`, `Snorlax`, `Charizard`...).

Résultat attendu : `chercher_ligne("pikachu")` → `Pikachu · type Electric · PV 35 · attaque 55 · défense 40 · vitesse 90 · total 320`.

<details><summary>Indice</summary>

`POKEMON["Name"].str.lower() == nom.strip().lower()` donne un masque de booléens. `lignes.empty` dit s'il n'y a rien, `lignes.iloc[0]` prend la première ligne trouvée, et `ligne["HP"]` une colonne.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def chercher_ligne(nom):
    """Cherche un Pokémon par son nom dans le tableau. Ex. chercher_ligne("Pikachu")."""
    nom = nom.strip()
    if not nom:
        return "Erreur : il me faut un nom de Pokémon."
    lignes = POKEMON[POKEMON["Name"].str.lower() == nom.lower()]
    if lignes.empty:
        return f"Erreur : aucun Pokémon nommé « {nom} » dans le tableau."
    l = lignes.iloc[0]
    return (f"{l['Name']} · type {l['Type 1']} · PV {l['HP']} · attaque {l['Attack']} "
            f"· défense {l['Defense']} · vitesse {l['Speed']} · total {l['Total']}")

print(chercher_ligne("pikachu"))
print(chercher_ligne("Zoubidou"))
```

</details>

In [ ]:
# À toi
def chercher_ligne(nom):
    """Cherche un Pokémon par son nom dans le tableau. Ex. chercher_ligne("Pikachu")."""
    return "Erreur : pas encore écrit."

print(chercher_ligne("pikachu"))
print(chercher_ligne("Zoubidou"))

In [ ]:
verifier("Exercice 2 · Pikachu est trouvé", lambda: "Pikachu" in chercher_ligne("Pikachu"))
verifier("Exercice 2 · la casse n'a pas d'importance", lambda: chercher_ligne("pikachu") == chercher_ligne("PIKACHU"))
verifier("Exercice 2 · ses statistiques sont dans la phrase", lambda: "35" in chercher_ligne("Pikachu") and "90" in chercher_ligne("Pikachu"))
verifier("Exercice 2 · un nom inconnu → Erreur", lambda: chercher_ligne("Zoubidou").startswith("Erreur"))
verifier("Exercice 2 · un nom vide → Erreur", lambda: chercher_ligne("").startswith("Erreur"))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if not str(chercher_ligne("Pikachu")).startswith("Pikachu"):
    def chercher_ligne(nom):
        nom = nom.strip()
        if not nom:
            return "Erreur : il me faut un nom de Pokémon."
        lignes = POKEMON[POKEMON["Name"].str.lower() == nom.lower()]
        if lignes.empty:
            return f"Erreur : aucun Pokémon nommé « {nom} » dans le tableau."
        l = lignes.iloc[0]
        return (f"{l['Name']} · type {l['Type 1']} · PV {l['HP']} · attaque {l['Attack']} "
                f"· défense {l['Defense']} · vitesse {l['Speed']} · total {l['Total']}")

### Outil 3 : la météo (simulée)

Troisième outil, le plus intéressant pédagogiquement : il **simule** un service extérieur. Pas de clé, pas de réseau, et surtout **déterministe** — la même ville donne toujours la même réponse, sinon les tests de la fin ne voudraient rien dire.

Le vrai appel est en commentaire juste en dessous : c'est le même outil, avec `requests` à la place de l'arithmétique.

### À toi · exercice 3 ⭐ · Une météo plausible et reproductible

Écris `meteo(ville)` : elle fabrique un bulletin à partir du **nom** de la ville. Le truc : additionner les codes des lettres (`sum(ord(c) for c in ville.lower())`) donne un nombre stable, dont on tire le ciel, la température et le vent.

Règles : une ville vide renvoie `Erreur : ...`, deux villes différentes donnent (presque toujours) deux bulletins différents, et deux appels sur la même ville donnent le même.

Résultat attendu : `meteo("Lyon")` → `À Lyon : ensoleillé, 18 °C, vent 5 km/h (météo simulée).`

<details><summary>Indice</summary>

`CIELS[graine % 5]` pour le ciel, `8 + graine % 20` pour les degrés, `5 + graine % 25` pour le vent. Pense à `ville.strip()` **avant** de calculer la graine.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
CIELS = ["ensoleillé", "nuageux", "pluvieux", "brumeux", "venteux"]

def meteo(ville):
    """Météo SIMULÉE et déterministe : la même ville donne toujours le même bulletin."""
    ville = ville.strip()
    if not ville:
        return "Erreur : il me faut un nom de ville."
    graine = sum(ord(c) for c in ville.lower())
    return (f"À {ville.title()} : {CIELS[graine % 5]}, {8 + graine % 20} °C, "
            f"vent {5 + graine % 25} km/h (météo simulée).")

print(meteo("Lyon"))
print(meteo("Brest"))
```

</details>

In [ ]:
# À toi
CIELS = ["ensoleillé", "nuageux", "pluvieux", "brumeux", "venteux"]

def meteo(ville):
    """Météo SIMULÉE et déterministe : la même ville donne toujours le même bulletin."""
    return "Erreur : pas encore écrit."

print(meteo("Lyon"))
print(meteo("Brest"))

In [ ]:
verifier("Exercice 3 · la ville est dans le bulletin", lambda: "Lyon" in meteo("Lyon"))
verifier("Exercice 3 · déterministe : deux appels, même bulletin", lambda: meteo("Lyon") == meteo(" lyon "))
verifier("Exercice 3 · deux villes → deux bulletins", lambda: meteo("Lyon") != meteo("Brest"))
verifier("Exercice 3 · une ville vide → Erreur", lambda: meteo("").startswith("Erreur"))
verifier("Exercice 3 · il y a des degrés dedans", lambda: "°C" in meteo("Lyon"))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if str(meteo("Lyon")).startswith("Erreur"):
    CIELS = ["ensoleillé", "nuageux", "pluvieux", "brumeux", "venteux"]
    def meteo(ville):
        ville = ville.strip()
        if not ville:
            return "Erreur : il me faut un nom de ville."
        graine = sum(ord(c) for c in ville.lower())
        return (f"À {ville.title()} : {CIELS[graine % 5]}, {8 + graine % 20} °C, "
                f"vent {5 + graine % 25} km/h (météo simulée).")

**En option : la vraie météo.** Open-Meteo est gratuit et sans clé. Décommente pour remplacer la simulation — le reste du notebook ne change pas d'une ligne, c'est tout l'intérêt d'avoir isolé l'outil derrière son nom.

```python
import requests
VILLES = {"lyon": (45.76, 4.84), "paris": (48.85, 2.35), "brest": (48.39, -4.49), "marseille": (43.30, 5.37)}

def meteo(ville):
    coord = VILLES.get(ville.strip().lower())
    if coord is None:
        return f"Erreur : je ne connais pas les coordonnées de « {ville} »."
    try:
        url = ("https://api.open-meteo.com/v1/forecast"
               f"?latitude={coord[0]}&longitude={coord[1]}&current_weather=true")
        m = requests.get(url, timeout=5).json()["current_weather"]
        return f"À {ville.title()} : {m['temperature']} °C, vent {m['windspeed']} km/h."
    except Exception as e:
        return f"Erreur : le service météo n'a pas répondu ({type(e).__name__})."
```

### Outil 4 : la date du jour

Le plus court, et pourtant indispensable : un modèle ne sait **pas** quel jour on est (sa réponse date de son entraînement). Celui-là est offert.

Note le `_=""` : tous nos outils reçoivent une chaîne, même ceux qui n'en ont pas besoin. Une signature commune, c'est ce qui permet de les appeler tous de la même façon dans la boucle.

In [ ]:
from datetime import date

JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin",
        "juillet", "août", "septembre", "octobre", "novembre", "décembre"]

def aujourdhui(_=""):
    """Renvoie la date du jour. Ex. aujourdhui() → 'mardi 10 septembre 2026'."""
    d = date.today()
    return f"{JOURS[d.weekday()]} {d.day} {MOIS[d.month - 1]} {d.year}"

print(aujourdhui())

### Le catalogue

Les quatre outils sont écrits et testés. On les range dans un dictionnaire : pour chacun, **la fonction** (que Python appellera) et **la description** (que le modèle lira). Ces deux champs sont la seule chose qui relie le modèle au code.

Une description vague donne un mauvais choix d'outil — on le vérifiera en section 4.

In [ ]:
OUTILS = {
    "calculer": {"fonction": calculer,
                 "description": "calculer(expression) — un calcul exact, exemple : calculer(17 * 23 + 5)"},
    "chercher_ligne": {"fonction": chercher_ligne,
                       "description": "chercher_ligne(nom) — les statistiques d'un Pokémon, exemple : chercher_ligne(Pikachu)"},
    "meteo": {"fonction": meteo,
              "description": "meteo(ville) — le temps qu'il fait dans une ville, exemple : meteo(Lyon)"},
    "aujourdhui": {"fonction": aujourdhui,
                   "description": "aujourdhui() — la date du jour, sans argument"},
}

for nom, outil in OUTILS.items():
    print(f"{nom:16} → {outil['fonction']('Pikachu' if nom == 'chercher_ligne' else 'Lyon' if nom == 'meteo' else '2 + 2')}")

## 3. La boucle d'agent

Le modèle ne peut pas appeler une fonction : il ne sait qu'écrire du texte. On lui donne donc une **convention d'écriture** — c'est ça, un protocole — et on se charge de la lire :

```
OUTIL: nom(arguments)
```

Le prompt système fait trois choses : il annonce le protocole, il liste les outils avec leur description, et il dit quoi faire une fois le résultat reçu. On le fabrique à partir du catalogue, pour qu'ajouter un outil suffise à le mettre à jour.

In [ ]:
def construire_systeme(outils, regle_en_plus=""):
    """Fabrique le prompt système à partir du catalogue d'outils."""
    catalogue = "\n".join("- " + o["description"] for o in outils.values())
    return ("Tu es un assistant qui peut utiliser des outils pour répondre juste.\n"
            "Quand tu as besoin d'un outil, écris sur une seule ligne, et rien d'autre :\n"
            "OUTIL: nom(arguments)\n"
            "Outils disponibles :\n" + catalogue + "\n"
            "Ensuite tu recevras « Résultat de l'outil : ... ». "
            "Rédige alors la réponse finale en une phrase, sans écrire OUTIL:.\n" + regle_en_plus)

SYSTEME = construire_systeme(OUTILS)
print(SYSTEME)

### À toi · exercice 4 ⭐⭐ · Repérer l'appel d'outil

Écris `extraire_appel(texte)` : elle renvoie le couple `(nom, arguments)` si le texte contient un appel `OUTIL: nom(args)`, et `None` sinon. Les arguments sont renvoyés **sans espaces autour**, et une ligne `OUTIL: aujourdhui()` donne des arguments vides.

Le modèle bavarde souvent avant d'appeler (« Je vais calculer. » puis la ligne) : cherche l'appel **n'importe où** dans le texte, pas seulement au début.

Résultat attendu : `("calculer", "2 + 2")`, puis `("aujourdhui", "")`, puis `None`.

<details><summary>Indice</summary>

Une seule expression régulière suffit : `re.search(r"OUTIL\s*:\s*(\w+)\s*\((.*)\)", texte)`. `.group(1)` est le nom, `.group(2)` les arguments. Le `.*` est gourmand : il s'arrête à la **dernière** parenthèse de la ligne, ce qui laisse passer `calculer(2 * (3 + 4))`.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
MOTIF_OUTIL = re.compile(r"OUTIL\s*:\s*(\w+)\s*\((.*)\)")

def extraire_appel(texte):
    """(nom, arguments) si le texte contient un appel OUTIL: nom(args), sinon None."""
    trouve = MOTIF_OUTIL.search(texte)
    if trouve is None:
        return None
    return trouve.group(1), trouve.group(2).strip()

for essai in ["OUTIL: calculer(2 + 2)", "Je vais regarder.\nOUTIL: aujourdhui()",
              "OUTIL: calculer(2 * (3 + 4))", "La réponse est 4."]:
    print(f"{essai!r:45} → {extraire_appel(essai)}")
```

</details>

In [ ]:
# À toi
def extraire_appel(texte):
    """(nom, arguments) si le texte contient un appel OUTIL: nom(args), sinon None."""
    return None

for essai in ["OUTIL: calculer(2 + 2)", "Je vais regarder.\nOUTIL: aujourdhui()",
              "OUTIL: calculer(2 * (3 + 4))", "La réponse est 4."]:
    print(f"{essai!r:45} → {extraire_appel(essai)}")

In [ ]:
verifier("Exercice 4 · un appel simple", lambda: extraire_appel("OUTIL: calculer(2 + 2)") == ("calculer", "2 + 2"))
verifier("Exercice 4 · un appel sans argument", lambda: extraire_appel("OUTIL: aujourdhui()") == ("aujourdhui", ""))
verifier("Exercice 4 · l'appel est trouvé même après du bavardage", lambda: extraire_appel("Je réfléchis.\nOUTIL: meteo(Lyon)") == ("meteo", "Lyon"))
verifier("Exercice 4 · les parenthèses imbriquées passent", lambda: extraire_appel("OUTIL: calculer(2 * (3 + 4))") == ("calculer", "2 * (3 + 4)"))
verifier("Exercice 4 · pas d'appel → None", lambda: extraire_appel("La réponse est 4.") is None)

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if extraire_appel("OUTIL: calculer(2 + 2)") != ("calculer", "2 + 2"):
    MOTIF_OUTIL = re.compile(r"OUTIL\s*:\s*(\w+)\s*\((.*)\)")
    def extraire_appel(texte):
        trouve = MOTIF_OUTIL.search(texte)
        return (trouve.group(1), trouve.group(2).strip()) if trouve else None

### À toi · exercice 5 ⭐⭐⭐ · Exécuter l'outil — sans jamais planter

Écris `executer_outil(nom, arguments)`. Elle renvoie **toujours** du texte, et jamais une exception : c'est elle qui protège la boucle.

Trois cas :
- le nom n'est pas dans `OUTILS` → `Erreur : l'outil « ... » n'existe pas.` **suivi de la liste des outils disponibles** (le modèle pourra se rattraper) ;
- la fonction lève une exception → `Erreur : ...` avec le type de l'exception ;
- sinon → le résultat, préfixé par `Résultat de l'outil : `. Si l'outil a lui-même renvoyé un `Erreur : ...`, on le laisse tel quel.

<details><summary>Indice</summary>

`if nom not in OUTILS: return ...` puis un `try/except Exception as e:`. Convertis toujours le résultat en texte avec `str(...)` — `calculer` renvoie un nombre.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def executer_outil(nom, arguments):
    """Exécute l'outil demandé et renvoie TOUJOURS du texte pour le modèle — jamais d'exception."""
    if nom not in OUTILS:
        return (f"Erreur : l'outil « {nom} » n'existe pas. "
                f"Outils disponibles : {', '.join(OUTILS)}.")
    try:
        resultat = str(OUTILS[nom]["fonction"](arguments))
    except Exception as e:
        return f"Erreur : l'outil {nom} a échoué ({type(e).__name__} : {e})."
    return resultat if resultat.startswith("Erreur") else "Résultat de l'outil : " + resultat

print(executer_outil("calculer", "17 * 23 + 5"))
print(executer_outil("wikipedia", "Pikachu"))
print(executer_outil("calculer", "10 / 0"))
```

</details>

In [ ]:
# À toi
def executer_outil(nom, arguments):
    """Exécute l'outil demandé et renvoie TOUJOURS du texte pour le modèle — jamais d'exception."""
    return None

print(executer_outil("calculer", "17 * 23 + 5"))
print(executer_outil("wikipedia", "Pikachu"))
print(executer_outil("calculer", "10 / 0"))

In [ ]:
verifier("Exercice 5 · un appel qui marche", lambda: executer_outil("calculer", "17 * 23 + 5") == "Résultat de l'outil : 396")
verifier("Exercice 5 · outil inconnu → Erreur, sans planter", lambda: executer_outil("wikipedia", "Pikachu").startswith("Erreur"))
verifier("Exercice 5 · l'erreur liste les outils disponibles", lambda: "calculer" in executer_outil("wikipedia", "Pikachu"))
verifier("Exercice 5 · l'erreur de l'outil passe telle quelle", lambda: executer_outil("calculer", "10 / 0").startswith("Erreur"))
verifier("Exercice 5 · arguments invalides → Erreur", lambda: executer_outil("chercher_ligne", "").startswith("Erreur"))
verifier("Exercice 5 · un outil sans argument marche", lambda: executer_outil("aujourdhui", "").startswith("Résultat"))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if executer_outil("calculer", "17 * 23 + 5") != "Résultat de l'outil : 396":
    def executer_outil(nom, arguments):
        if nom not in OUTILS:
            return (f"Erreur : l'outil « {nom} » n'existe pas. "
                    f"Outils disponibles : {', '.join(OUTILS)}.")
        try:
            resultat = str(OUTILS[nom]["fonction"](arguments))
        except Exception as e:
            return f"Erreur : l'outil {nom} a échoué ({type(e).__name__} : {e})."
        return resultat if resultat.startswith("Erreur") else "Résultat de l'outil : " + resultat

### À toi · exercice 6 ⭐⭐⭐ · La boucle, bornée et tracée

Il reste à tourner en rond — proprement. La boucle appelle le modèle, regarde s'il demande un outil, l'exécute, **remet le résultat dans la conversation** et recommence. Deux sorties possibles : le modèle répond sans demander d'outil (c'est fini), ou la borne `max_tours` est atteinte (on abandonne, sans boucler à l'infini). Cette borne n'est pas un détail : un modèle qui redemande le même outil en boucle brûlerait ton quota d'API en quelques secondes.

Écris `agent(question, systeme=SYSTEME, max_tours=5, trace=True)`.

Le déroulé d'un tour : appeler `llm(messages, temperature=0)`, afficher ce que le modèle dit si `trace`, extraire l'appel — s'il n'y en a pas, **renvoyer la réponse** —, sinon exécuter l'outil, afficher le résultat, puis ajouter **deux** messages à la conversation : celui du modèle (`assistant`) et le résultat (`user`).

Si les `max_tours` tours sont épuisés, renvoie une phrase contenant le mot `limite`.

<details><summary>Indice</summary>

```python
messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
for tour in range(1, max_tours + 1):
    reponse = llm(messages, temperature=0)
    ...
```
Le résultat de l'outil repart en `{"role": "user", ...}` : pour le modèle, c'est un nouveau message qui arrive de l'extérieur.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def agent(question, systeme=SYSTEME, max_tours=5, trace=True):
    """Boucle d'agent : le modèle demande un outil, on l'exécute, on lui rend le résultat."""
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    for tour in range(1, max_tours + 1):
        reponse = llm(messages, temperature=0)
        if trace:
            print(f"— tour {tour} · le modèle dit : {reponse}")
        appel = extraire_appel(reponse)
        if appel is None:
            return reponse
        resultat = executer_outil(appel[0], appel[1])
        if trace:
            print(f"           outil exécuté  : {appel[0]}({appel[1]}) → {resultat}")
        messages.append({"role": "assistant", "content": reponse})
        messages.append({"role": "user", "content": resultat})
    return f"J'abandonne : la limite de {max_tours} tour(s) est atteinte."

print(agent("Combien font 17 * 23 + 5 ?"))
```

</details>

In [ ]:
# À toi
def agent(question, systeme=SYSTEME, max_tours=5, trace=True):
    """Boucle d'agent : le modèle demande un outil, on l'exécute, on lui rend le résultat."""
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    for tour in range(1, max_tours + 1):
        pass    # appeler le modèle, extraire l'appel, exécuter, remettre le résultat dans messages
    return f"J'abandonne : la limite de {max_tours} tour(s) est atteinte."

print(agent("Combien font 17 * 23 + 5 ?"))

In [ ]:
verifier("Exercice 6 · le calcul passe par l'outil", lambda: "396" in agent("Combien font 17 * 23 + 5 ?", trace=False))
verifier("Exercice 6 · la boucle s'arrête dès que le modèle répond", lambda: "OUTIL:" not in agent("Quel temps fait-il à Lyon ?", trace=False))
verifier("Exercice 6 · la réponse reprend le résultat de l'outil", lambda: "Lyon" in agent("Quel temps fait-il à Lyon ?", trace=False))
verifier("Exercice 6 · la borne est respectée", lambda: "limite" in agent("Quel temps fait-il à Lyon ?", max_tours=1, trace=False))
verifier("Exercice 6 · une question sans outil ne plante pas", lambda: isinstance(agent("Qui est Pierre de Coubertin ?", trace=False), str))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if "396" not in str(agent("Combien font 17 * 23 + 5 ?", trace=False)):
    def agent(question, systeme=SYSTEME, max_tours=5, trace=True):
        messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
        for tour in range(1, max_tours + 1):
            reponse = llm(messages, temperature=0)
            if trace:
                print(f"— tour {tour} · le modèle dit : {reponse}")
            appel = extraire_appel(reponse)
            if appel is None:
                return reponse
            resultat = executer_outil(appel[0], appel[1])
            if trace:
                print(f"           outil exécuté  : {appel[0]}({appel[1]}) → {resultat}")
            messages += [{"role": "assistant", "content": reponse}, {"role": "user", "content": resultat}]
        return f"J'abandonne : la limite de {max_tours} tour(s) est atteinte."

La trace complète de deux questions. C'est elle qu'on regarde quand un agent se comporte mal : elle montre **ce que le modèle a dit**, **quel outil est parti**, et **ce qui lui est revenu**.

In [ ]:
for question in ["Quelle est la vitesse de Pikachu ?", "Quel temps fait-il à Brest ?"]:
    print("=" * 70)
    print("QUESTION :", question)
    print("RÉPONSE  :", agent(question))

## 4. Le choix de l'outil

L'agent tourne. Reste la vraie question : **choisit-il le bon outil ?** On le mesure comme on mesure un modèle — avec un jeu de test. Six questions de types différents, et pour chacune l'outil qu'on attend.

On ne regarde ici que le **premier** appel : c'est là que la décision se joue.

In [ ]:
def premier_outil(question, systeme=SYSTEME):
    """Quel outil le modèle demande-t-il en premier ? '(aucun)' s'il répond directement."""
    reponse = llm([{"role": "system", "content": systeme}, {"role": "user", "content": question}], temperature=0)
    appel = extraire_appel(reponse)
    return appel[0] if appel else "(aucun)"

QUESTIONS_TEST = [
    ("Combien font 17 * 23 + 5 ?", "calculer"),
    ("Quelle est la vitesse de Pikachu ?", "chercher_ligne"),
    ("Quel temps fait-il à Lyon ?", "meteo"),
    ("Quelle est la date du jour ?", "aujourdhui"),
    ("Faut-il un parapluie à Brest ?", "meteo"),
    ("Qui est Pierre de Coubertin ?", "(aucun)"),
]
print(len(QUESTIONS_TEST), "questions de test")

In [ ]:
def tableau_des_choix(questions, systeme=SYSTEME):
    lignes = []
    for question, attendu in questions:
        choisi = premier_outil(question, systeme)
        lignes.append({"question": question, "outil attendu": attendu,
                       "outil choisi": choisi, "": "✅" if choisi == attendu else "❌"})
    return pd.DataFrame(lignes)

choix = tableau_des_choix(QUESTIONS_TEST)
bons = (choix["outil attendu"] == choix["outil choisi"]).sum()
print(f"{bons}/{len(choix)} bons choix d'outil")
choix

**Lis la ligne qui échoue.** « Qui est Pierre de Coubertin ? » n'a de réponse dans aucun de nos quatre outils — et le modèle invente un outil `wikipedia` qui n'existe pas. Ce n'est pas un caprice : notre prompt système décrit ce qu'il faut faire *avec* un outil, mais ne dit nulle part quoi faire **quand aucun outil ne convient**. Le modèle comble le vide.

C'est le réflexe à prendre : quand un agent choisit mal, on relit **la description des outils et le prompt système** avant de blâmer le modèle. Ici, une phrase de plus suffit.

In [ ]:
REGLE_SANS_OUTIL = ("Si aucun outil ne convient à la question, n'invente pas d'outil : "
                    "réponds directement, et dis que tu n'as pas pu vérifier.")

SYSTEME_V2 = construire_systeme(OUTILS, REGLE_SANS_OUTIL)

choix_v2 = tableau_des_choix(QUESTIONS_TEST, systeme=SYSTEME_V2)
bons_v2 = (choix_v2["outil attendu"] == choix_v2["outil choisi"]).sum()
print(f"prompt d'origine : {bons}/{len(choix)}  →  prompt corrigé : {bons_v2}/{len(choix_v2)}")
choix_v2

## 5. Ce qui rate

Un agent passe le plus clair de son temps dans les cas tordus. Quatre pannes possibles, et la même règle pour toutes : **l'erreur repart au modèle sous forme de texte**, la cellule ne plante pas.

**Panne 1 — l'outil n'existe pas.** Le message d'erreur lui rappelle la liste : c'est ce qui lui donne une chance de se rattraper.
**Panne 2 — les arguments sont invalides.** L'outil sait dire non lui-même : c'est pour ça que chacun renvoie un `Erreur : ...` lisible plutôt que de lever une exception.

In [ ]:
print(executer_outil("wikipedia", "Pierre de Coubertin"))
print(executer_outil("chercher_ligne", ""))
print(executer_outil("chercher_ligne", "Zoubidou"))
print(executer_outil("calculer", "import os"))
print(executer_outil("calculer", "(2 + 3"))
print(executer_outil("calculer", "10 / 0"))

**Panne 3 — l'outil lève une exception.** Celle-là, on ne peut pas la prévoir : un bug, une API qui répond n'importe quoi. Le `try/except` de `executer_outil` est le dernier rempart. On le vérifie avec un outil cassé exprès.

In [ ]:
def outil_casse(_=""):
    """Un outil volontairement bogué, juste pour vérifier que la boucle survit."""
    return 1 / 0

OUTILS["outil_casse"] = {"fonction": outil_casse, "description": "outil_casse() — outil de test, il plante exprès"}
print(executer_outil("outil_casse", ""))
del OUTILS["outil_casse"]
print("la cellule est arrivée jusqu'ici : rien n'a planté ✅")

**Panne 4 — la boucle n'en finit pas.** Un modèle qui redemande sans cesse un outil tournerait à l'infini. `max_tours` coupe. On force le cas avec `max_tours=1` : le modèle demande la météo, on l'exécute… et il n'a plus de tour pour rédiger.

**Et le rattrapage.** Voilà pourquoi on renvoie les erreurs en texte plutôt que de planter : le modèle les lit et peut changer d'avis. Sur la deuxième trace, il appelle un outil qui n'existe pas, reçoit la liste des outils, tente autre chose, échoue encore — et finit par dire honnêtement qu'il n'a pas trouvé, au lieu d'inventer. Trois tours, aucune exception.

In [ ]:
print(agent("Quel temps fait-il à Nantes ?", max_tours=1))
print("=" * 70)
print(agent("Qui est Pierre de Coubertin ?", systeme=SYSTEME))

## 6. Ton outil, et le banc de test

Quatre outils, ça suffit pour un agent qui marche. Le cinquième, c'est le tien — et c'est la partie du projet qui te ressemble.

Idées : les 3 Pokémon les plus forts d'un type, le nombre de jours avant une date, convertir des euros en dollars à un taux fixe, tirer un élément au hasard dans une liste, compter les mots d'un texte… La règle ne change pas : une chaîne en entrée, du texte en sortie, un `Erreur : ...` quand ça coince, et **une description claire** dans le catalogue.

### À toi · exercice 7 ⭐⭐⭐ · Ajoute ton outil

Écris ton outil, puis **enregistre-le dans `OUTILS`** avec sa description — sans ça, le modèle ne saura pas qu'il existe.

L'outil de référence proposé ici est `top_type(type_pokemon)` : les 3 Pokémon les plus puissants d'un type (`Fire`, `Water`, `Grass`, `Psychic`...). Remplace-le par le tien si tu as une meilleure idée — les vérifications ci-dessous portent sur `top_type`, adapte-les à ton outil dans ce cas.

<details><summary>Indice</summary>

`POKEMON[POKEMON["Type 1"].str.lower() == t]` filtre le tableau, `.nlargest(3, "Total")` garde les trois plus forts, et `.iterrows()` permet de fabriquer la phrase.

</details>

<details><summary>Solution (à ouvrir après avoir essayé)</summary>

```python
def top_type(type_pokemon):
    """Les 3 Pokémon les plus puissants d'un type. Ex. top_type("Fire")."""
    t = type_pokemon.strip().lower()
    if not t:
        return "Erreur : il me faut un type (Fire, Water, Grass...)."
    lignes = POKEMON[POKEMON["Type 1"].str.lower() == t]
    if lignes.empty:
        return f"Erreur : aucun Pokémon de type « {type_pokemon} »."
    return ", ".join(f"{l['Name']} ({l['Total']})" for _, l in lignes.nlargest(3, "Total").iterrows())

OUTILS["top_type"] = {"fonction": top_type,
                      "description": "top_type(type) — les 3 Pokémon les plus forts d'un type, exemple : top_type(Fire)"}
print(top_type("Fire"))
```

</details>

In [ ]:
# À toi
def top_type(type_pokemon):
    """Les 3 Pokémon les plus puissants d'un type. Ex. top_type("Fire")."""
    return "Erreur : pas encore écrit."

OUTILS["top_type"] = {"fonction": top_type,
                      "description": "top_type(type) — les 3 Pokémon les plus forts d'un type, exemple : top_type(Fire)"}
print(top_type("Fire"))

In [ ]:
verifier("Exercice 7 · l'outil répond pour le type Fire", lambda: not top_type("Fire").startswith("Erreur"))
verifier("Exercice 7 · un type inconnu → Erreur", lambda: top_type("Zzzz").startswith("Erreur"))
verifier("Exercice 7 · un type vide → Erreur", lambda: top_type("").startswith("Erreur"))
verifier("Exercice 7 · l'outil est enregistré dans le catalogue", lambda: "top_type" in OUTILS)
verifier("Exercice 7 · il est décrit dans le prompt système", lambda: "top_type" in construire_systeme(OUTILS))

In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Replié comme une solution : il réinjecte l'implémentation de référence,
# pour que la suite du notebook fonctionne quand même.
if str(top_type("Fire")).startswith("Erreur"):
    def top_type(type_pokemon):
        t = type_pokemon.strip().lower()
        if not t:
            return "Erreur : il me faut un type (Fire, Water, Grass...)."
        lignes = POKEMON[POKEMON["Type 1"].str.lower() == t]
        if lignes.empty:
            return f"Erreur : aucun Pokémon de type « {type_pokemon} »."
        return ", ".join(f"{l['Name']} ({l['Total']})" for _, l in lignes.nlargest(3, "Total").iterrows())
    OUTILS["top_type"] = {"fonction": top_type,
                          "description": "top_type(type) — les 3 Pokémon les plus forts d'un type, exemple : top_type(Fire)"}

Le catalogue a changé : on reconstruit le prompt système, et l'agent connaît le nouvel outil.

En mode démo (`USE_MODEL = False`), le faux modèle de la séance 12 ne connaît que quatre outils. On lui ajoute **une branche**, sans toucher à la cellule de préparation, exactement comme on l'avait fait pour le RAG : si la question parle du « plus fort » d'un type, il appelle `top_type`. Avec `USE_MODEL = True`, cette branche ne sert jamais.

In [ ]:
SYSTEME_V3 = construire_systeme(OUTILS, REGLE_SANS_OUTIL)
_llm_factice_base = llm_factice                  # on garde la version de la section 0

def llm_factice(messages):
    systeme = " ".join(m["content"] for m in messages if m["role"] == "system")
    question = next((m["content"] for m in messages if m["role"] == "user"), "")
    dernier = messages[-1]["content"]
    if ("top_type" in systeme and "plus fort" in question.lower()
            and not dernier.startswith(("Résultat", "Erreur"))):
        return "OUTIL: top_type(%s)" % _nom_propre(question)
    return _llm_factice_base(messages)            # tout le reste : la section 0

print(agent("Quel est le Pokémon le plus fort de type Fire ?", systeme=SYSTEME_V3))

### Le banc de test

Sept questions, la réponse finale de l'agent, et un mot qu'on doit y trouver. C'est le chiffre à mettre dans ta fiche projet — et le seul moyen de savoir si une modification du prompt améliore les choses ou les empire.

In [ ]:
BANC = [
    {"question": "Combien font 17 * 23 + 5 ?", "outil": "calculer", "attendu": "396"},
    {"question": "Quelle est la vitesse de Pikachu ?", "outil": "chercher_ligne", "attendu": "90"},
    {"question": "Quel temps fait-il à Lyon ?", "outil": "meteo", "attendu": "Lyon"},
    {"question": "Quelle est la date du jour ?", "outil": "aujourdhui", "attendu": str(date.today().year)},
    {"question": "Combien font 10 / 0 ?", "outil": "calculer", "attendu": "division par zéro"},
    {"question": "Qui est Pierre de Coubertin ?", "outil": "(aucun)", "attendu": "mémoire"},
    {"question": "Quelle est la somme des points de vie de Pikachu et de Snorlax ?", "outil": "calculer", "attendu": "195"},
]

def passer_le_banc(banc, systeme=SYSTEME_V3):
    """Fait tourner l'agent sur chaque cas et renvoie (taux de réussite, détail)."""
    detail = []
    for cas in banc:
        reponse = agent(cas["question"], systeme=systeme, trace=False)
        detail.append({**cas, "reponse": reponse, "ok": cas["attendu"].lower() in reponse.lower()})
    return sum(d["ok"] for d in detail) / len(detail), detail

taux, detail = passer_le_banc(BANC)
for d in detail:
    print(("✅" if d["ok"] else "❌"), d["question"], f"(attendu : {d['attendu']})")
    print("    →", d["reponse"][:110])
print(f"\n=== {taux:.0%} de réussite ({sum(d['ok'] for d in detail)}/{len(detail)}) ===")

In [ ]:
noms = [d["question"][:28] + "…" for d in detail]
plt.figure(figsize=(7, 3))
plt.barh(noms, [1 if d["ok"] else 0 for d in detail],
         color=["tab:green" if d["ok"] else "tab:red" for d in detail])
plt.xlim(0, 1.15)
plt.xticks([0, 1], ["raté", "réussi"])
plt.title(f"Banc de test de l'agent : {taux:.0%}")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**L'échec à expliquer.** Le dernier cas rate, et il rate pour une bonne raison : « la somme des points de vie de Pikachu **et** de Snorlax » demande **deux outils enchaînés** — `chercher_ligne` deux fois, puis `calculer` pour additionner. Notre agent ne fait qu'un aller-retour : il cherche une ligne, reçoit un résultat, et rédige. Il ne se dit jamais « il me manque encore quelque chose ».

Trois façons de le corriger, par ordre de coût :

1. **Un outil de plus** : `somme_pv("Pikachu, Snorlax")` fait le travail en un appel. Rapide, mais il faudra un outil par question tordue.
2. **Un prompt système qui autorise l'enchaînement** : « tu peux appeler plusieurs outils l'un après l'autre ; ne réponds que lorsque tu as tout ». La boucle est déjà prête (elle tourne jusqu'à `max_tours`) — c'est le modèle qu'il faut convaincre, et un modèle de 0,5 milliard de paramètres y arrive mal.
3. **Un plan explicite** : demander d'abord la liste des étapes, puis les exécuter une par une. C'est le patron *plan-and-execute*, et c'est ce que font les vrais agents.

Note aussi le cas 5 : `10 / 0` échoue, mais l'agent **dit** qu'il a échoué au lieu d'inventer un nombre. C'est un ✅, et c'est le comportement qu'on veut.

## Conclusion et fiche projet

À retenir :

- **Un outil est une fonction Python testée seule.** Une chaîne en entrée, du texte en sortie, jamais d'exception qui remonte. Tout ce qui n'a pas été testé sans agent sera indébogable avec.
- **Jamais `eval()` sur le texte d'un modèle.** `ast` coûte quinze lignes et enlève le risque.
- **Le protocole est une convention, pas de la magie.** `OUTIL: nom(args)` et une expression régulière suffisent ; le JSON du *function calling* n'est que la version robuste de la même idée.
- **La boucle doit être bornée.** `max_tours` est ce qui sépare un agent d'une facture d'API.
- **Un mauvais choix d'outil est presque toujours un défaut de description.** On corrige le prompt système, pas le modèle.

Remplis la fiche ci-dessous : c'est ce que tu montreras.

In [ ]:
MA_SYNTHESE = """(à remplir) Mon agent a ... outils, dont ... que j'ai écrit. Il choisit bien l'outil quand ... ,
et il se trompe sur ... parce que ... . Pour le corriger, je ... ."""

print("=== FICHE PROJET B2 · UN AGENT QUI SAIT CHOISIR SON OUTIL ===")
print(f"Outils            : {len(OUTILS)} → {', '.join(OUTILS)}")
print(f"Données           : {len(POKEMON)} lignes ({SOURCE})")
print(f"Choix de l'outil  : {bons}/{len(choix)} avec le prompt d'origine, {bons_v2}/{len(choix_v2)} avec le prompt corrigé")
print(f"Banc de test      : {taux:.0%} ({sum(d['ok'] for d in detail)}/{len(detail)} cas)")
print(f"Cas raté          : {[d['question'] for d in detail if not d['ok']] or 'aucun'}")
print(f"Mode              : {'modèle Qwen2.5-0.5B' if USE_MODEL else 'démo (llm_factice)'}")
print("\nMa synthèse :", MA_SYNTHESE)

## Pour aller plus loin

- **La vraie météo** : remplace la simulation par l'appel **Open-Meteo** (cellule en commentaire de la section 2) et gère les cas réels — ville inconnue, réseau coupé, réponse lente. Le `timeout=5` de `requests` n'est pas décoratif : sans lui, ton agent peut rester bloqué une minute sur un appel.
- **Passe au JSON** : `{"outil": "calculer", "arguments": {"expression": "17 * 23 + 5"}}` au lieu de `OUTIL: nom(args)`. Plus robuste à parser (`json.loads` dans un `try`), et c'est exactement ce que font les API de *function calling*.
- **Enchaîne deux outils** : c'est l'échec du banc de test. Reprends la piste 2 ou 3 de l'analyse ci-dessus et remesure le banc — un chiffre avant, un chiffre après, c'est le meilleur graphique de fin de projet.
- **Un agent qui appelle un RAG** : branche la recherche du projet [B1](../B1-assistant-reviseur/) comme un cinquième outil `chercher_dans_mes_notes`. Un agent qui interroge un RAG, c'est l'architecture de la plupart des assistants d'entreprise.

Liens utiles : le modèle https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct · le module `ast` https://docs.python.org/fr/3/library/ast.html · les expressions régulières https://docs.python.org/fr/3/library/re.html · l'API météo https://open-meteo.com/en/docs · ce qu'est un agent en 40 pages https://www.kaggle.com/whitepaper-agents · des patrons d'agents simples https://www.anthropic.com/engineering/building-effective-agents · le *function calling* côté Kaggle https://www.kaggle.com/code/markishere/day-3-function-calling-with-the-gemini-api